# 07 - Prompt Engineering and Generation

## Definition
Prompt engineering defines behavior contracts for grounded, citation-aware generation.

## Theory
Instruction clarity and constraints reduce unsupported claims and improve consistency.

## Motivation
A high-recall retriever still fails if prompts allow context-ignoring behavior.

## Architecture
Retrieved context + policy instructions + question -> constrained generation.

## Real-world Examples
- Enterprise support assistant over product docs
- Compliance assistant over policy text
- Research assistant over paper abstracts

## Best Practices
- Measure retrieval and generation separately
- Track latency and grounding together
- Keep prompts strict about citations and uncertainty

## Common Mistakes
- Skipping retrieval diagnostics
- Using mismatched embedding models for index/query
- Reporting only one metric and claiming broad quality


In [1]:
from pathlib import Path
import json
import pandas as pd

from rag_system import (
    RAGConfig,
    ProjectRunner,
    ChunkingStrategy,
)

cfg = RAGConfig(project_root=Path(".."), profile="balanced")
runner = ProjectRunner(cfg)


In [2]:
from rag_system.prompts import PromptLibrary

query = 'Explain retrieval-augmented generation in simple terms.'
context = '[1] RAG retrieves relevant documents before generation.'

bad = PromptLibrary.bad_prompt_example(query, context)
good = PromptLibrary.good_prompt_example(query, context)
bad[:300], good[:300]


('Answer anything user asks quickly. Maybe use context if useful. Question: Explain retrieval-augmented generation in simple terms.\nContext: [1] RAG retrieves relevant documents before generation.',
 'Role: retrieval-grounded assistant.\nConstraints: only use context, cite evidence, admit uncertainty.\nContext:\n[1] RAG retrieves relevant documents before generation.\n\nQuestion: Explain retrieval-augmented generation in simple terms.\nOutput: concise factual answer with citation markers [i].')

In [3]:
bundle = runner.ingest_dataset()
chunking = runner.run_chunking(bundle['documents'][:650], strategy=ChunkingStrategy.RECURSIVE)
runner.index_chunks(chunking.chunks, reset=True)

result = runner.pipeline.answer(bundle['queries'][10].query, top_k=6)
{
    'answer': result.answer[:420],
    'citations': result.citations,
    'abstained': result.abstained,
    'abstain_reason': result.abstain_reason,
}


{'answer': "I cannot find enough evidence in retrieved documents to answer who ruled the Duchy of Normandy, as none of the provided texts mention this topic or any rulers associated with it [1], [2], [3]. The available context focuses on New York City's history and Sino-Tibetan relations during the Ming dynasty.",
 'citations': ['[1] doc=doc_3a785144fa2dfbfb title=New_York_City score=0.722',
  '[2] doc=doc_64b70c59b8bf800d title=New_York_City score=0.711',
  '[3] doc=doc_abafec42b57d3f74 title=New_York_City score=0.705',
  '[4] doc=doc_6021b4277dcc821b title=Sino-Tibetan_relations_during_the_Ming_dynasty score=0.695',
  '[5] doc=doc_771433c531df1a4f title=2008_Sichuan_earthquake score=0.690',
  '[6] doc=doc_81c5d551e73eff48 title=Sino-Tibetan_relations_during_the_Ming_dynasty score=0.689'],
 'abstained': False,
 'abstain_reason': ''}

## Code Explanation
The prompt library enforces context-only answering and explicit abstention when evidence is insufficient.